<p align="center">
  <a href="https://colab.research.google.com/github/samuel-c-santos/satellite-lst-downscaler/blob/main/LST_Downscaling_IDEFLORBio_Colab.ipynb" target="_blank">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Google Colab" width="180">
  </a>
</p>

# Satellite LST Downscaler - Desagregacao Termica para Gestao Territorial
## Monitoramento de Temperatura da Superficie Terrestre (LST) em Areas do IDEFLOR-Bio

Este caderno implementa a metodologia de **downscaling termico (Thermal Sharpening)** para desagregar dados orbitais de Temperatura da Superficie Terrestre (LST) de resolucao quilometrica para escalas compativeis com a analise de paisagem (10 a 30 metros), integrando preditores multiespectrais do sensor Sentinel-2 ou Landsat.

### Principais Aplicacoes na Gestao de Unidades de Conservacao:
- **Efeito de Borda e Amortecimento Termico**: Mapeamento do gradiente de aquecimento entre pastagens/matriz antropica externa e o macico florestal protegido.
- **Manejo Florestal e Concessoes**: Diagnostico de estresse hidrico no dossel e identificacao de microclimas especificos.
- **Resiliencia e Vulnerabilidade ao Fogo**: Mapeamento de anomalias termicas de superficie em periodos de estiagem prolongada.
- **Camadas Nativas do IDEFLOR-Bio**: Selecao interativa direta de Unidades de Conservacao Estaduais, Concessoes Florestais e Centro de Treinamento Agroflorestal.


## 1. Instalacao das Dependencias
Instala bibliotecas de processamento geoespacial, modelagem estatistica e interface no Google Colab.

In [ ]:
# Instalacao de pacotes geoespaciais e de machine learning
!pip install -q geemap earthengine-api geopandas rasterio scikit-learn scipy matplotlib ipywidgets

## 2. Autenticacao e Inicializacao (Earth Engine e Google Drive)
Inicializa o projeto `sentinel-landsat-download` e monta o Google Drive para leitura dos vetores e gravacao dos produtos termicos.

In [ ]:
import ee
import os
import shutil
from google.colab import drive

# 1. Inicializacao do Google Earth Engine
GEE_PROJECT = "sentinel-landsat-download"
try:
    ee.Initialize(project=GEE_PROJECT)
    print(f"Google Earth Engine inicializado no projeto: {GEE_PROJECT}")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print(f"Google Earth Engine autenticado e inicializado: {GEE_PROJECT}")

# 2. Montagem do Google Drive
drive.mount("/content/drive")
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/Satellite_Downloads/LST_Downscaled"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"Diretorio de saida no Google Drive: {DRIVE_OUTPUT_DIR}")

# 3. Diretorios locais de trabalho
WORK_DIR = "/content/data_lst"
os.makedirs(WORK_DIR, exist_ok=True)

# Variaveis globais de regiao e nome
SELECTED_GEOMETRY = None
SELECTED_AOI_NAME = "Area_IDEFLORBio"

## 3. Selecao da Area de Atuacao do IDEFLOR-Bio
Selecione uma das camadas do IDEFLOR-Bio disponiveis no projeto (Unidades de Conservacao Estaduais, Concessoes Florestais, Contratos de Transicao ou Centro de Treinamento) ou faca upload de um novo Shapefile compactado em `.zip`.

In [ ]:
import os
import glob
import json as _json
import zipfile
import subprocess
import urllib.request
import geopandas as gpd
import ipywidgets as widgets
from IPython.display import display, clear_output
from shapely import force_2d

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 1. Localizacao ou sincronizacao automatica das camadas vetoriais do IDEFLOR-Bio
POSSIBLE_DATA_DIRS = [
    "./data",
    "../data",
    "/content/data",
    "/content/satellite-lst-downscaler/data",
    "/content/drive/MyDrive/DEV/downscale-satelliteLST/data"
]

DATA_DIR = "./data"
for p in POSSIBLE_DATA_DIRS:
    if os.path.exists(p) and len(glob.glob(os.path.join(p, "*.geojson"))) > 0:
        DATA_DIR = p
        break

DATA_DIR = os.path.abspath(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)
sample_check = os.path.join(DATA_DIR, "ucs_estaduais.geojson")

# Se rodando no Google Colab ou se a pasta data nao existir localmente,
# faz o download automatico dos vetores diretamente do repositorio oficial:
if not os.path.exists(sample_check):
    print("Dados vetoriais nao identificados localmente. Sincronizando com o repositorio GitHub...")
    repo_url = "https://github.com/samuel-c-santos/satellite-lst-downscaler.git"
    cloned_ok = False
    try:
        temp_dir = "/content/_temp_repo" if os.path.exists("/content") else "./_temp_repo"
        subprocess.run(["git", "clone", "--depth", "1", repo_url, temp_dir], check=True, capture_output=True)
        src_data = os.path.join(temp_dir, "data")
        if os.path.exists(src_data):
            import shutil
            for f in os.listdir(src_data):
                shutil.copy(os.path.join(src_data, f), os.path.join(DATA_DIR, f))
            shutil.rmtree(temp_dir, ignore_errors=True)
            cloned_ok = True
            print("Dados geoespaciais sincronizados com sucesso via Git!")
    except Exception:
        cloned_ok = False

    if not cloned_ok:
        print("Baixando camadas GeoJSON diretamente do GitHub...")
        raw_base = "https://raw.githubusercontent.com/samuel-c-santos/satellite-lst-downscaler/main/data"
        geojson_files = [
            "ucs_estaduais.geojson",
            "concessoes_florestais_ideflor.geojson",
            "contratos_de_transicao.geojson",
            "centro_treinamento.geojson"
        ]
        for gf in geojson_files:
            dst = os.path.join(DATA_DIR, gf)
            if not os.path.exists(dst):
                try:
                    urllib.request.urlretrieve(f"{raw_base}/{gf}", dst)
                    print(f"  -> Camada obtida: {gf}")
                except Exception as err:
                    print(f"  -> Falha ao baixar {gf}: {err}")

print(f"Pasta de dados geoespaciais ativa: {DATA_DIR}")

# 2. Variaveis Globais de Geometria
SELECTED_BOUNDS = None
loaded_gdf = None

# 3. Componentes da Interface (Widgets)
output_aoi = widgets.Output()

layer_selector = widgets.Dropdown(
    options=[
        ("Unidades de Conservacao Estaduais (29 UCs)", "ucs_estaduais.geojson"),
        ("Concessoes Florestais IDEFLOR-Bio (11 areas)", "concessoes_florestais_ideflor.geojson"),
        ("Contratos de Transicao (43 areas)", "contratos_de_transicao.geojson"),
        ("Centro de Treinamento Agroflorestal", "centro_treinamento.geojson"),
        ("Fazer Upload de Outro Shapefile (.zip)", "UPLOAD")
    ],
    value="ucs_estaduais.geojson",
    description="Camada Base:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="560px")
)

feature_dropdown = widgets.Dropdown(
    options=[],
    description="Unidade/Alvo:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="560px")
)

upload_btn = widgets.Button(
    description="Fazer Upload do Shapefile (.zip)",
    button_style="info",
    icon="upload",
    layout=widgets.Layout(width="300px", margin="4px 0 4px 135px")
)

confirm_btn = widgets.Button(
    description="Confirmar Area de Estudo",
    button_style="success",
    icon="check",
    layout=widgets.Layout(width="300px", margin="6px 0 6px 135px")
)

# Container dinamico: exibe a lista suspensa OU o botao de upload (nunca ambos)
target_slot = widgets.Box([feature_dropdown])

def apply_selected_geometry(idx):
    global SELECTED_GEOMETRY, SELECTED_AOI_NAME, SELECTED_BOUNDS
    if loaded_gdf is None or idx is None or idx >= len(loaded_gdf):
        return
    geom = loaded_gdf.geometry.iloc[idx]
    SELECTED_GEOMETRY = ee.Geometry(geom.__geo_interface__)
    name_label = [opt[0] for opt in feature_dropdown.options if opt[1] == idx]
    SELECTED_AOI_NAME = str(name_label[0]).replace(" ", "_").replace("/", "_") if name_label else f"Area_{idx}"
    SELECTED_BOUNDS = geom.bounds
    minx, miny, maxx, maxy = SELECTED_BOUNDS
    print(f"Alvo selecionado: {SELECTED_AOI_NAME}")
    print(f"Extensao geografica: Longitude [{minx:.4f}, {maxx:.4f}] | Latitude [{miny:.4f}, {maxy:.4f}]")
    print("Clique no botao verde 'Confirmar Area de Estudo' para fixar a selecao.")

def update_features_list(change):
    global loaded_gdf
    selected_file = change["new"] if isinstance(change, dict) else getattr(change, "new", change)
    with output_aoi:
        clear_output()
        if selected_file == "UPLOAD":
            # Troca o slot dinamicamente para o botao de upload
            target_slot.children = [upload_btn]
            print("Modo de Upload ativado. Clique no botao 'Fazer Upload do Shapefile (.zip)' acima.")
            return

        # Restaura o slot com o dropdown de alvos
        target_slot.children = [feature_dropdown]

        filepath = os.path.join(DATA_DIR, selected_file)
        if not os.path.exists(filepath):
            print(f"Arquivo nao encontrado em {filepath}.")
            return

        loaded_gdf = gpd.read_file(filepath)
        if loaded_gdf.crs is None:
            loaded_gdf = loaded_gdf.set_crs("EPSG:4326")
        elif loaded_gdf.crs.to_string() != "EPSG:4326":
            loaded_gdf = loaded_gdf.to_crs("EPSG:4326")

        try:
            loaded_gdf["geometry"] = loaded_gdf["geometry"].apply(force_2d)
        except Exception:
            pass

        name_col = None
        for col in ["Nome", "NOME", "NOME_", "Categoria", "CAT_UC"]:
            if col in loaded_gdf.columns:
                name_col = col
                break

        if name_col:
            options_list = [(str(row[name_col]), idx) for idx, row in loaded_gdf.iterrows()]
        else:
            options_list = [(f"Poligono #{idx+1}", idx) for idx in range(len(loaded_gdf))]

        feature_dropdown.unobserve(on_feature_selected, names="value")
        feature_dropdown.options = options_list
        if options_list:
            feature_dropdown.value = options_list[0][1]
            apply_selected_geometry(options_list[0][1])
        feature_dropdown.observe(on_feature_selected, names="value")

def on_feature_selected(change):
    idx = change["new"] if isinstance(change, dict) else getattr(change, "new", change)
    with output_aoi:
        clear_output()
        apply_selected_geometry(idx)

def upload_shp_action(b):
    global SELECTED_GEOMETRY, SELECTED_AOI_NAME, SELECTED_BOUNDS
    if not IN_COLAB:
        with output_aoi:
            clear_output()
            print("Upload interativo disponivel em ambiente Google Colab.")
        return
    with output_aoi:
        clear_output()
        print("Selecione o arquivo .zip com o Shapefile...")
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.lower().endswith(".zip"):
                extract_dir = os.path.join(WORK_DIR, "shp_upload")
                os.makedirs(extract_dir, exist_ok=True)
                with zipfile.ZipFile(fname, "r") as zr:
                    zr.extractall(extract_dir)
                shps = [f for f in os.listdir(extract_dir) if f.lower().endswith(".shp")]
                if shps:
                    shp_f = os.path.join(extract_dir, shps[0])
                    gdf_up = gpd.read_file(shp_f)
                    if gdf_up.crs is None:
                        gdf_up = gdf_up.set_crs("EPSG:4326")
                    elif gdf_up.crs.to_string() != "EPSG:4326":
                        gdf_up = gdf_up.to_crs("EPSG:4326")
                    try:
                        gdf_up["geometry"] = gdf_up["geometry"].apply(force_2d)
                    except Exception:
                        pass
                    SELECTED_GEOMETRY = ee.Geometry(gdf_up.unary_union.__geo_interface__)
                    SELECTED_AOI_NAME = os.path.splitext(shps[0])[0]
                    SELECTED_BOUNDS = gdf_up.unary_union.bounds
                    minx, miny, maxx, maxy = SELECTED_BOUNDS
                    print(f"Shapefile carregado com sucesso: {SELECTED_AOI_NAME}")
                    print(f"Extensao: Longitude [{minx:.4f}, {maxx:.4f}] | Latitude [{miny:.4f}, {maxy:.4f}]")
                    print("Clique no botao verde 'Confirmar Area de Estudo' para finalizar.")
                    return
        print("Nenhum arquivo .shp valido encontrado no .zip.")

def on_confirm_clicked(b):
    with output_aoi:
        clear_output()
        if SELECTED_GEOMETRY is None:
            print("Atencao: Nenhuma geometria selecionada ou carregada.")
            return
        print(f" AREA DE ESTUDO CONFIRMADA: {SELECTED_AOI_NAME}")
        if SELECTED_BOUNDS:
            minx, miny, maxx, maxy = SELECTED_BOUNDS
            print(f"  Extensao Geografica: Longitude [{minx:.4f}, {maxx:.4f}] | Latitude [{miny:.4f}, {maxy:.4f}]")
        print("Pronto! Voce ja pode prosseguir e executar a Celula 4.")

# 4. Registro de Eventos e Exibicao Organizada
layer_selector.observe(update_features_list, names="value")
feature_dropdown.observe(on_feature_selected, names="value")
upload_btn.on_click(upload_shp_action)
confirm_btn.on_click(on_confirm_clicked)

# Layout em VBox estruturado com espacamento uniforme
ui_container = widgets.VBox([
    layer_selector,
    target_slot,
    confirm_btn,
    output_aoi
], layout=widgets.Layout(gap="10px", padding="10px 0"))

display(ui_container)
update_features_list({"new": layer_selector.value})


## 4. Aquisicao dos Dados Orbitais via Google Earth Engine
Coleta simultanea no mesmo periodo sobre a regiao de interesse:
1. **Sinal Termico de Superficie**: Colecao `MODIS/061/MOD11A1` (LST Diurna em Kelvin convertida para graus Celsius a 1.000m).
2. **Preditores de Alta Resolucao**: Colecao Sentinel-2 L2A (`COPERNICUS/S2_SR_HARMONIZED`), calculando o indice de vegetacao por diferenca normalizada (NDVI) a 10 metros.

In [ ]:
import urllib.request
import rasterio
import numpy as np

if SELECTED_GEOMETRY is None:
    raise ValueError("Selecione uma area de interesse na celula 3 antes de prosseguir.")

# Intervalo de amostragem sazonal (periodo seco / menor nebulosidade na Amazonia)
DATA_INICIO = "2025-08-01"
DATA_FIM = "2025-11-30"

aoi_bounds = SELECTED_GEOMETRY.bounds()

print(f"Consultando Earth Engine para {SELECTED_AOI_NAME} entre {DATA_INICIO} e {DATA_FIM}...")

# 1. LST Baixa Resolucao (MODIS 1km Diurno - Kelvin para Celsius)
modis_col = (
    ee.ImageCollection("MODIS/061/MOD11A1")
    .filterBounds(SELECTED_GEOMETRY)
    .filterDate(DATA_INICIO, DATA_FIM)
    .select("LST_Day_1km")
)
# Multiplicador de escala MODIS e conversao de Kelvin para Celsius
lst_1km_celsius = modis_col.median().multiply(0.02).subtract(273.15).clip(aoi_bounds)

# 2. Preditores de Alta Resolucao (Sentinel-2 10m: NDVI, B4, B3, B2, B8)
s2_col = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(SELECTED_GEOMETRY)
    .filterDate(DATA_INICIO, DATA_FIM)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
    .sort("CLOUDY_PIXEL_PERCENTAGE")
)

if s2_col.size().getInfo() == 0:
    print("Nenhuma cena Sentinel-2 com < 20% de nuvens. Utilizando composicao mediana do periodo...")
    s2_col = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(SELECTED_GEOMETRY)
        .filterDate(DATA_INICIO, DATA_FIM)
    )

s2_composite = s2_col.median()
ndvi_10m = s2_composite.normalizedDifference(["B8", "B4"]).rename("NDVI")
predictors_10m = ndvi_10m.addBands(s2_composite.select(["B4", "B3", "B2", "B8"])).clip(aoi_bounds)

# 3. Download das matrizes para processamento local
lst_tif_path = os.path.join(WORK_DIR, f"lst_1km_{SELECTED_AOI_NAME}.tif")
pred_tif_path = os.path.join(WORK_DIR, f"predictors_10m_{SELECTED_AOI_NAME}.tif")

# Estimativa da dimensao da regiao para ajustar a escala e respeitar os limites de download do Earth Engine
try:
    coords = aoi_bounds.coordinates().get(0).getInfo()
    lons = [p[0] for p in coords]
    lats = [p[1] for p in coords]
    dx_m = (max(lons) - min(lons)) * 111320 * np.cos(np.radians(np.mean(lats)))
    dy_m = (max(lats) - min(lats)) * 111320
    approx_pixels_30m = (dx_m / 30.0) * (dy_m / 30.0)
except Exception:
    approx_pixels_30m = 1_000_000
    dx_m, dy_m = 30000, 30000

if approx_pixels_30m > 20_000_000:
    pred_scale = 120
    print(f"Area de dimensao continental (~{dx_m/1e3:.0f}x{dy_m/1e3:.0f} km). Escala dos preditores ajustada para {pred_scale}m para download seguro.")
elif approx_pixels_30m > 8_000_000:
    pred_scale = 60
    print(f"Area muito extensa (~{dx_m/1e3:.0f}x{dy_m/1e3:.0f} km). Escala dos preditores ajustada para {pred_scale}m.")
else:
    pred_scale = 30
    print(f"Escala dos preditores definida em {pred_scale}m.")

url_lst = lst_1km_celsius.getDownloadURL({
    "scale": 1000,
    "crs": "EPSG:3857",
    "region": aoi_bounds,
    "format": "GEO_TIFF"
})

url_pred = predictors_10m.getDownloadURL({
    "scale": pred_scale,
    "crs": "EPSG:3857",
    "region": aoi_bounds,
    "format": "GEO_TIFF"
})

print("Baixando dados termicos e preditores multiespectrais...")
urllib.request.urlretrieve(url_lst, lst_tif_path)
urllib.request.urlretrieve(url_pred, pred_tif_path)

print("Dados baixados com sucesso para o ambiente de execucao.")


## 5. Algoritmo de Downscaling Termico (Thermal Sharpening com Ajuste de Residuos)
A metodologia modela a relacao funcional entre a Temperatura da Superficie Terrestre (LST) e os preditores multiespectrais em baixa resolucao por regressao com Random Forest. O modelo e aplicado na grade fina e os residuos sao interpolados espacialmente, preservando o balanco termico de conservacao da energia radiativa:

$$\text{LST}_{\text{fina}} = f(\text{Preditores}_{\text{finos}}) + \Delta\text{LST}_{\text{interpolado}}$$

In [ ]:
from scipy.ndimage import zoom
from sklearn.ensemble import RandomForestRegressor

# 1. Leitura das matrizes raster
with rasterio.open(lst_tif_path) as src_lst:
    lst_lr = src_lst.read(1)
    lst_profile = src_lst.profile

with rasterio.open(pred_tif_path) as src_pred:
    pred_hr = src_pred.read()  # (Bandas, H, W)
    pred_profile = src_pred.profile

print(f"Matriz LST Baixa Resolucao (1km): {lst_lr.shape}")
print(f"Matriz Preditores Alta Resolucao: {pred_hr.shape}")

# 2. Reamostragem dos preditores para a grade de baixa resolucao para calibracao
n_bands, h_hr, w_hr = pred_hr.shape
h_lr, w_lr = lst_lr.shape

zoom_y = h_lr / h_hr
zoom_x = w_lr / w_hr

pred_lr = np.zeros((n_bands, h_lr, w_lr), dtype=np.float32)
for b in range(n_bands):
    pred_lr[b] = zoom(pred_hr[b], (zoom_y, zoom_x), order=1)[:h_lr, :w_lr]

# 3. Preparacao das amostras para treinamento
valid_mask = (~np.isnan(lst_lr)) & (lst_lr > 0) & (lst_lr < 70)
for b in range(n_bands):
    valid_mask &= (~np.isnan(pred_lr[b]))

X_train = pred_lr[:, valid_mask].T
y_train = lst_lr[valid_mask]

print(f"Amostras validas para calibracao estatistica: {len(y_train):,} pixels")

# 4. Ajuste do modelo estatistico de Random Forest
rf_model = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
r2_score = rf_model.score(X_train, y_train)
print(f"Coeficiente de determinacao (R2) do modelo calibrado: {r2_score:.3f}")

# 5. Predicao na baixa resolucao e calculo do campo de residuos
pred_lst_lr = np.full_like(lst_lr, np.nan)
pred_lst_lr[valid_mask] = rf_model.predict(X_train)
residuals_lr = np.where(valid_mask, lst_lr - pred_lst_lr, 0.0)

# 6. Interpolação suave dos residuos para a grade fina
res_zoom_y = h_hr / h_lr
res_zoom_x = w_hr / w_lr
residuals_hr = zoom(residuals_lr, (res_zoom_y, res_zoom_x), order=1)[:h_hr, :w_hr]

# 7. Aplicacao do modelo aos preditores de alta resolucao e soma residual
X_hr = pred_hr.reshape(n_bands, -1).T
X_hr_clean = np.nan_to_num(X_hr, nan=0.0)

lst_hr_baseline = rf_model.predict(X_hr_clean).reshape(h_hr, w_hr)
lst_downscaled = lst_hr_baseline + residuals_hr

print("Downscaling termico finalizado com sucesso.")
print(f"Temperatura media desagregada na regiao: {np.nanmean(lst_downscaled):.2f} graus Celsius")

## 6. Gravacao do GeoTIFF de LST Desagregada e Exportacao para o Google Drive
Gera o arquivo matricial com temperatura em graus Celsius (°C) e georreferenciamento completo na pasta do Google Drive.

In [ ]:
out_profile = pred_profile.copy()
out_profile.update({
    "count": 1,
    "dtype": "float32",
    "nodata": -9999.0
})

out_filename = f"LST_Downscaled_{SELECTED_AOI_NAME}.tif"
local_output = os.path.join(WORK_DIR, out_filename)
drive_output = os.path.join(DRIVE_OUTPUT_DIR, out_filename)

# Gravacao local
with rasterio.open(local_output, "w", **out_profile) as dst:
    dst.write(lst_downscaled.astype(np.float32), 1)
print(f"GeoTIFF local gerado: {local_output}")

# Exportacao para o Google Drive
shutil.copy2(local_output, drive_output)
print(f"Arquivo copiado com sucesso para o Google Drive:")
print(f"  {drive_output}")

## 7. Inspecao Visual Comparativa e Gradiente Termico da Paisagem
Comparacao visual entre o dado termico bruto de 1 km, os preditores de vegetacao e a Temperatura da Superficie Terrestre desagregada em escala de paisagem.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. LST Bruta 1km (Interpolada apenas para visualizacao)
vmin = np.nanpercentile(lst_lr[valid_mask], 2)
vmax = np.nanpercentile(lst_lr[valid_mask], 98)

im0 = axes[0].imshow(lst_lr, cmap="inferno", vmin=vmin, vmax=vmax)
axes[0].set_title(f"LST Original 1km (MODIS)\n{SELECTED_AOI_NAME}", fontsize=11, fontweight="bold")
axes[0].axis("off")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label="Temperatura (C)")

# 2. Indice de Vegetacao (NDVI em alta resolucao)
ndvi_img = pred_hr[0]
im1 = axes[1].imshow(ndvi_img, cmap="YlGn", vmin=0.1, vmax=0.85)
axes[1].set_title("Preditor de Biomassa (NDVI Alta Resolucao)\nSentinel-2 / Landsat", fontsize=11, fontweight="bold")
axes[1].axis("off")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, label="NDVI")

# 3. LST Downscaled Desagregada (Alta Resolucao)
im2 = axes[2].imshow(lst_downscaled, cmap="inferno", vmin=vmin, vmax=vmax)
axes[2].set_title(f"LST Desagregada (Downscaled)\nDetalhamento Espacial de Paisagem", fontsize=11, fontweight="bold", color="darkred")
axes[2].axis("off")
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04, label="Temperatura (C)")

plt.tight_layout()
plt.show()